# Replay a T-Rex Dataset episode on the real URDFs

Local + heavy: needs the `[replay]` extra, `pinocchio` (conda-forge), `dexmate_urdf` (arms), and the vendored Sharpa hand under `third_party/`. Plays one episode's 58-D `observation.state` on the Dexmate + Sharpa Pinocchio model in viser.

The dataset order `[L_arm | L_hand | R_arm | R_hand]` is bridged to the Pinocchio qpos order **only** via `schema.split_state` -> `assemble_qpos` (never positionally).

In [1]:
import time

import viser
from pinocchio.visualize import ViserVisualizer
from IPython.display import IFrame, display

from trex_dataset_quickstart import decode, hub, robot, schema

## Config

In [ ]:
SOURCE = "zekaiwang/trex_dataset"  # HF repo id, or a path to a local dataset dir
EPISODE_INDEX = 1234
FPS = schema.FPS

## Load the episode state (tiny pull — no videos)

In [3]:
root = hub.fetch_episode(SOURCE, EPISODE_INDEX, include_videos=False)
episodes = hub.load_episodes(root)
rows = episodes[episodes["episode_index"] == EPISODE_INDEX]
if len(rows) == 0:
    raise ValueError(f"episode_index {EPISODE_INDEX} not found in {SOURCE}")
length = int(rows.iloc[0]["length"])

state = decode.load_episode_state(root, EPISODE_INDEX, length)
state.shape

(751, 58)

## Build the robot model (Dexmate arms + Sharpa hands, and a table)

In [4]:
combined_robot, assemble_qpos, _ = robot.build_full_robot(
    default_joint_by_component={"head": robot.DEFAULT_HEAD, "torso": robot.DEFAULT_TORSO}
)
combined_robot = robot.add_table_model(
    robot=combined_robot,
    default_joint_by_component=schema.split_state(state[0]),
    assemble_qpos=assemble_qpos,
    table_height=robot.TABLE_HEIGHT,
)

## Start the viewer

In [5]:
server = viser.ViserServer()
server.scene.add_box(
    "floor",
    dimensions=(20.0, 20.0, 0.01),
    position=(0, -0.005, 0),
    color=(190, 150, 255),
    opacity=1,
)
viz = ViserVisualizer(
    combined_robot.model,
    combined_robot.collision_model,
    combined_robot.visual_model,
    copy_models=True,
)
viz.initViewer(viewer=server, open=False, loadModel=False)
viz.loadViewerModel(rootNodeName="robot")
viz.displayCollisions(False)
viz.displayVisuals(True)

# Show the viewer inline (or open the printed http://localhost:<port> URL in a browser).
display(IFrame(src=f"http://localhost:{server.get_port()}", width="100%", height="640"))

╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

## Play

Re-run this cell to replay. Drive a faded `action` overlay the same way if you want target-vs-actual.

In [6]:
dt = 1.0 / FPS
for s in state:
    viz.display(assemble_qpos(schema.split_state(s)))
    time.sleep(dt)

(viser) Connection opened (0, 1 total), 1064 persistent messages

## Stop the server when done

In [7]:
server.stop()

(viser) Connection closed (0, 0 total)

(viser) Server stopped